# Praperation

In [ ]:
!pip install torch transformers huggingface_hub

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
import torch

torch.cuda.is_available()

True

In [4]:
torch.cuda.get_device_name(0)

'Tesla T4'

In [5]:
torch.cuda.device_count()

1

# Accessing by Pipeline

In [6]:
from transformers import pipeline

# A default model will be selected when not explicitly passing a model's name
classifier = pipeline("sentiment-analysis")
classifier(
    ["I love deeplearning!",
     "I hate it so much!"]
)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9996706247329712},
 {'label': 'NEGATIVE', 'score': 0.9995473027229309}]

In [7]:
# In the case of text generation models, pipeline will default to GPT-2 Model
text_generator = pipeline("text-generation")

text_generator(
    ["I went to this store to buy",
     "When two objects in space get close to each other"]
)

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[[{'generated_text': "I went to this store to buy a $10 bill and it was a little bit different than what I was expecting. It's kind of like the Starbucks you can buy for $20, but they have different things. I got a really nice bar. It's like Starbucks. It's really good. It's got a little bit of a lot of spice that is nice to have in it.\n\nI'm not sure if I would have been able to bring it back to work, but I've been to Starbucks and I've gotten a beer. I've never gotten a beer.\n\nWhat do you do with beer that you find on campus?\n\nI'm just a kid. I live in a small town in Ohio. I came to Ohio and I went to college. I'm not a big drinker. I do an average of five beers a week. I would have to go to college for that. I think I'm a great student.\n\nWhat do you do when you're not drinking beer?\n\nI don't drink beer. I don't do a lot of drinking. I don't do a lot of drinking. I have a lot of good friends. I can't drive anymore, so I can't go out to the grocery store.\n\nHave"}],
 [{'gen

In [8]:
# Use a pipeline as a high-level helper
# Warning: Pipeline type "summarization" is no longer supported in transformers v5.
# You must load the model directly (see below) or downgrade to v4.x with:
# 'pip install "transformers<5.0.0'

# summarizer = pipeline("summarization") # not work

# summarizer("""
# The Fibonacci numbers were first described in Indian mathematics as early as 200 BC in work by Pingala on enumerating possible patterns of Sanskrit poetry formed from syllables of two lengths.[3][4][5] They are named after the Italian mathematician Leonardo of Pisa, also known as Fibonacci, who introduced the sequence to Western European mathematics in his 1202 book Liber Abaci.[6]
# Fibonacci numbers appear unexpectedly often in mathematics, so much so that there is an entire journal dedicated to their study, the Fibonacci Quarterly. Applications of Fibonacci numbers include computer algorithms such as the Fibonacci search technique and the Fibonacci heap data structure, and graphs called Fibonacci cubes used for interconnecting parallel and distributed systems. They also appear in biological settings, such as branching in trees, the arrangement of leaves on a stem, the fruit sprouts of a pineapple, the flowering of an artichoke, and the arrangement of a pine cone's bracts, though they do not occur in all species.
# """)

# Accessing Pre-Trained Models Directly

In [9]:
# Comparing with accessing the same model in pipeline
# Accessing model by pipeline does not need tokenizer, and will give you nicely formatted results
generator = pipeline("text-classification", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
generator("I love deep learning")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998319149017334}]

In [10]:
# Accessing pre-trained model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english")
model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [11]:
inputs = tokenizer("I love deep learning", return_tensors="pt")
inputs

{'input_ids': tensor([[ 101, 1045, 2293, 2784, 4083,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}

In [12]:
# When accessing models directly, it will not give you POSSITIVE or NEGATIVE
# Instead, you will have logits and do softmax calculation yourself
outputs = model(**inputs)
outputs

SequenceClassifierOutput(loss=None, logits=tensor([[-4.1975,  4.4937]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

# Model Embeddings

In [13]:
from transformers import AutoModel

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english")
model = AutoModel.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english")

inputs = tokenizer("I love deep learning", padding=True, truncation=True, return_tensors="pt")
outputs = model(**inputs)

# 1 sentences
# 7 tokens in total
# 768 dimensions for each token
print(outputs.last_hidden_state.shape) # the token embedding

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.weight | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 6, 768])


In [14]:
# To get the full context vector for the sequence
# By simply averaging the 7 tokens
# Converts 7 vectors to 1 vector
context_vectors = outputs.last_hidden_state.mean(dim=1)
context_vectors.shape

torch.Size([1, 768])

# Accessing Model Config & Creating Custom Models

In [15]:
from transformers import GPT2Config, GPT2Model

# Building the config
config = GPT2Config()

print(config)

GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "use_cache": true,
  "vocab_size": 50257
}



In [16]:
# Building the model from the config
# transformers will build a fresh, randomly intialized, from scratch GPT-2 Models

gpt_model = GPT2Model(config)

In [17]:
# Save model to your disks
gpt_model.save_pretrained("gpt2_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]